# DWTS Bayesian Fan Vote Estimation (2026 MCM Problem C)

This notebook implements a Bayesian model to estimate weekly fan vote shares for DWTS contestants, using judges' scores and elimination outcomes. It follows the specification provided in the prompt and produces:

- Posterior estimates of fan vote shares (and scaled vote counts)
- Uncertainty intervals (90% CI)
- Elimination consistency metrics per season

The workflow is organized into functional steps:

- `load_and_preprocess()`
- `build_season_data(season)`
- `log_posterior(params, season_data)`
- `metropolis_sampler(...)`
- `posterior_to_estimates(...)`
- `evaluate_consistency(...)`


In [ ]:
import json
import math
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd


In [ ]:
DATA_PATH = Path('2026_MCM_Problem_C_Data.csv')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

RNG = np.random.default_rng(42)


In [ ]:
def load_and_preprocess(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    score_cols = [c for c in df.columns if 'week' in c and 'judge' in c]
    for col in score_cols:
        df[col] = pd.to_numeric(df[col].replace({'N/A': np.nan, '': np.nan}), errors='coerce')
    return df


df = load_and_preprocess(DATA_PATH)
df.head()


In [ ]:
def extract_week_info(columns):
    week_indices = []
    for col in columns:
        if 'week' in col and 'judge' in col:
            # column format: weekX_judgeY_score
            try:
                week_str = col.split('_')[0].replace('week', '')
                week_indices.append(int(week_str))
            except ValueError:
                continue
    return sorted(set(week_indices))


def build_season_data(df: pd.DataFrame, season: int):
    season_df = df[df['season'] == season].copy()
    score_cols = [c for c in season_df.columns if 'week' in c and 'judge' in c]
    week_indices = extract_week_info(score_cols)
    t_max = max(week_indices)

    # Build judges total matrix J[i, t]
    contestants = season_df['celebrity_name'].tolist()
    n = len(contestants)
    j_scores = np.zeros((n, t_max))

    for t in range(1, t_max + 1):
        week_cols = [c for c in score_cols if c.startswith(f'week{t}_')]
        if not week_cols:
            continue
        week_data = season_df[week_cols].to_numpy(dtype=float)
        j_scores[:, t - 1] = np.nan_to_num(week_data, nan=0.0).sum(axis=1)

    active_mask = j_scores > 0
    t_last = np.where(active_mask.any(axis=1), active_mask[:, ::-1].argmax(axis=1), t_max)
    t_last = t_max - t_last  # last positive week index (1-based)

    elim_weeks = {}
    for t in range(1, t_max + 1):
        elim_weeks[t] = [i for i, tl in enumerate(t_last) if tl == t and t < t_max]

    return {
        'season': season,
        'contestants': contestants,
        'j_scores': j_scores,
        't_max': t_max,
        'elim_weeks': elim_weeks,
    }


season_data = build_season_data(df, season=1)
season_data['season'], season_data['t_max'], len(season_data['contestants'])


In [ ]:
def softmax_logits(logits):
    max_logit = np.max(logits)
    exp_vals = np.exp(logits - max_logit)
    return exp_vals / np.sum(exp_vals)


def rank_scores(values):
    # Higher value -> lower rank number (1 = best)
    series = pd.Series(values)
    return series.rank(ascending=False, method='average').to_numpy()


def compute_week_votes(j_scores, theta, alpha):
    active_idx = np.where(j_scores > 0)[0]
    if len(active_idx) == 0:
        return np.array([]), np.array([]), np.array([])

    j_active = j_scores[active_idx]
    mean_j = np.mean(j_active)
    sd_j = np.std(j_active)
    x = (j_active - mean_j) / (sd_j + 1e-6)

    u = theta[active_idx] + alpha * x
    p = softmax_logits(u)
    return active_idx, x, p


def elimination_likelihood_percent(j_scores, p, lam):
    q = j_scores / j_scores.sum()
    c = q + p
    logits = -lam * c
    probs = softmax_logits(logits)
    return probs


def elimination_likelihood_rank(j_scores, p, lam):
    r_j = rank_scores(j_scores)
    r_f = rank_scores(p)
    s = r_j + r_f
    logits = lam * s
    probs = softmax_logits(logits)
    return probs, s


def pair_elim_prob(probs, idx1, idx2):
    p1 = probs[idx1]
    p2 = probs[idx2]
    p1_then = p1 * (p2 / (1 - p1 + 1e-12))
    p2_then = p2 * (p1 / (1 - p2 + 1e-12))
    return p1_then + p2_then


In [ ]:
def log_prior(params):
    theta = params['theta']
    sigma_theta = params['sigma_theta']
    alpha = params['alpha']
    lam = params['lambda']
    kappa = params.get('kappa', None)

    if sigma_theta <= 0 or lam <= 0 or (kappa is not None and kappa <= 0):
        return -np.inf

    lp = 0.0
    # sigma_theta ~ HalfNormal(1)
    lp += -0.5 * (sigma_theta ** 2)
    # theta ~ Normal(0, sigma_theta)
    lp += -0.5 * np.sum((theta / sigma_theta) ** 2) - len(theta) * np.log(sigma_theta)
    # alpha ~ Normal(0,1)
    lp += -0.5 * (alpha ** 2)
    # lambda ~ HalfNormal(5)
    lp += -0.5 * (lam / 5) ** 2
    # kappa ~ HalfNormal(5)
    if kappa is not None:
        lp += -0.5 * (kappa / 5) ** 2

    return lp


def log_posterior(params, season_data):
    theta = params['theta']
    alpha = params['alpha']
    lam = params['lambda']
    kappa = params.get('kappa', None)

    j_scores_matrix = season_data['j_scores']
    elim_weeks = season_data['elim_weeks']
    t_max = season_data['t_max']
    season = season_data['season']

    lp = log_prior(params)
    if not np.isfinite(lp):
        return -np.inf

    ll = 0.0
    for t in range(1, t_max + 1):
        elim_idx = elim_weeks.get(t, [])
        if len(elim_idx) == 0:
            continue

        active_idx, _, p = compute_week_votes(j_scores_matrix[:, t - 1], theta, alpha)
        if len(active_idx) == 0:
            continue

        j_active = j_scores_matrix[active_idx, t - 1]

        if season <= 2 or season >= 28:
            probs, s_vals = elimination_likelihood_rank(j_active, p, lam)
        else:
            probs = elimination_likelihood_percent(j_active, p, lam)

        if len(elim_idx) == 1:
            elim_pos = np.where(active_idx == elim_idx[0])[0]
            if elim_pos.size == 0:
                ll += np.log(1e-12)
            else:
                ll += np.log(probs[elim_pos[0]] + 1e-12)
        else:
            elim_positions = [np.where(active_idx == idx)[0] for idx in elim_idx]
            elim_positions = [pos[0] for pos in elim_positions if pos.size > 0]
            if len(elim_positions) < 2:
                ll += np.log(1e-12)
            else:
                p_pair = pair_elim_prob(probs, elim_positions[0], elim_positions[1])
                ll += np.log(p_pair + 1e-12)

        if season >= 28 and kappa is not None:
            # Judges save model on bottom-2
            bottom2 = np.argsort(s_vals)[-2:]
            a, b = bottom2[0], bottom2[1]
            prob_elim_a = 1 / (1 + np.exp(-kappa * (j_active[b] - j_active[a])))
            elim_positions = [np.where(active_idx == idx)[0] for idx in elim_idx]
            elim_positions = [pos[0] for pos in elim_positions if pos.size > 0]
            if len(elim_positions) == 1:
                elim_obs = elim_positions[0]
                if elim_obs == a:
                    ll += np.log(prob_elim_a + 1e-12)
                elif elim_obs == b:
                    ll += np.log(1 - prob_elim_a + 1e-12)
                else:
                    ll += np.log(1e-9)

    return lp + ll


In [ ]:
def metropolis_sampler(season_data, n_iter=2000, burn=0.3, thin=2, seed=42):
    rng = np.random.default_rng(seed)
    n = len(season_data['contestants'])

    params = {
        'theta': rng.normal(0, 0.5, size=n),
        'sigma_theta': 1.0,
        'alpha': 0.5,
        'lambda': 2.0,
    }
    if season_data['season'] >= 28:
        params['kappa'] = 2.0

    proposal_sd = {
        'theta': 0.2,
        'sigma_theta': 0.1,
        'alpha': 0.1,
        'lambda': 0.2,
        'kappa': 0.2,
    }

    current_lp = log_posterior(params, season_data)
    samples = []
    accept = 0

    for step in range(n_iter):
        proposal = {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in params.items()}
        proposal['theta'] = params['theta'] + rng.normal(0, proposal_sd['theta'], size=n)
        proposal['sigma_theta'] = abs(params['sigma_theta'] + rng.normal(0, proposal_sd['sigma_theta']))
        proposal['alpha'] = params['alpha'] + rng.normal(0, proposal_sd['alpha'])
        proposal['lambda'] = abs(params['lambda'] + rng.normal(0, proposal_sd['lambda']))
        if 'kappa' in params:
            proposal['kappa'] = abs(params['kappa'] + rng.normal(0, proposal_sd['kappa']))

        prop_lp = log_posterior(proposal, season_data)
        if np.log(rng.uniform()) < prop_lp - current_lp:
            params = proposal
            current_lp = prop_lp
            accept += 1

        if step > n_iter * burn and step % thin == 0:
            samples.append({k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in params.items()})

        if step > 0 and step % 500 == 0:
            acc_rate = accept / step
            if acc_rate < 0.2:
                for k in proposal_sd:
                    proposal_sd[k] *= 0.9
            elif acc_rate > 0.4:
                for k in proposal_sd:
                    proposal_sd[k] *= 1.1

    return samples


In [ ]:
def posterior_to_estimates(season_data, samples, total_votes=10_000_000):
    n = len(season_data['contestants'])
    t_max = season_data['t_max']
    p_samples = np.zeros((len(samples), n, t_max))

    for m, params in enumerate(samples):
        theta = params['theta']
        alpha = params['alpha']
        for t in range(1, t_max + 1):
            active_idx, _, p = compute_week_votes(season_data['j_scores'][:, t - 1], theta, alpha)
            if len(active_idx) == 0:
                continue
            p_samples[m, active_idx, t - 1] = p

    p_mean = p_samples.mean(axis=0)
    p_low = np.quantile(p_samples, 0.05, axis=0)
    p_high = np.quantile(p_samples, 0.95, axis=0)

    v_mean = p_mean * total_votes
    v_low = p_low * total_votes
    v_high = p_high * total_votes

    rows = []
    for i, name in enumerate(season_data['contestants']):
        for t in range(1, t_max + 1):
            rows.append({
                'season': season_data['season'],
                'week': t,
                'celebrity_name': name,
                'p_mean': p_mean[i, t - 1],
                'p_ci_low': p_low[i, t - 1],
                'p_ci_high': p_high[i, t - 1],
                'V_mean': v_mean[i, t - 1],
                'V_ci_low': v_low[i, t - 1],
                'V_ci_high': v_high[i, t - 1],
            })

    return pd.DataFrame(rows)


def evaluate_consistency(season_data, samples):
    t_max = season_data['t_max']
    elim_weeks = season_data['elim_weeks']
    season = season_data['season']

    probs = []
    deterministic_correct = 0
    total_elim_weeks = 0

    for t in range(1, t_max + 1):
        elim_idx = elim_weeks.get(t, [])
        if len(elim_idx) == 0:
            continue

        total_elim_weeks += 1
        week_probs = []
        for params in samples:
            theta = params['theta']
            alpha = params['alpha']
            lam = params['lambda']

            active_idx, _, p = compute_week_votes(season_data['j_scores'][:, t - 1], theta, alpha)
            if len(active_idx) == 0:
                continue

            j_active = season_data['j_scores'][active_idx, t - 1]
            if season <= 2 or season >= 28:
                probs_rank, s_vals = elimination_likelihood_rank(j_active, p, lam)
                week_probs.append(probs_rank)
            else:
                probs_pct = elimination_likelihood_percent(j_active, p, lam)
                week_probs.append(probs_pct)

        if not week_probs:
            continue

        mean_probs = np.mean(week_probs, axis=0)
        elim_positions = [np.where(active_idx == idx)[0] for idx in elim_idx]
        elim_positions = [pos[0] for pos in elim_positions if pos.size > 0]

        if len(elim_positions) == 1:
            pi_t = mean_probs[elim_positions[0]]
        else:
            pi_t = pair_elim_prob(mean_probs, elim_positions[0], elim_positions[1])

        probs.append(pi_t)

        predicted = active_idx[np.argmax(mean_probs)]
        if predicted in elim_idx:
            deterministic_correct += 1

    if total_elim_weeks == 0:
        return {
            'accuracy': None,
            'mean_pi': None,
            'min_pi': None,
            'logscore': None,
        }

    probs_arr = np.array(probs)
    return {
        'accuracy': deterministic_correct / total_elim_weeks,
        'mean_pi': probs_arr.mean(),
        'min_pi': probs_arr.min(),
        'logscore': np.mean(np.log(probs_arr + 1e-12)),
    }


In [ ]:
# Example run for a single season (adjust n_iter as needed)
SEASON_TO_RUN = 1
season_data = build_season_data(df, season=SEASON_TO_RUN)

samples = metropolis_sampler(season_data, n_iter=1000, burn=0.3, thin=2, seed=42)

fan_estimates = posterior_to_estimates(season_data, samples)
summary = evaluate_consistency(season_data, samples)

fan_csv = OUTPUT_DIR / f'fan_vote_estimates_season_{SEASON_TO_RUN}.csv'
summary_json = OUTPUT_DIR / f'season_{SEASON_TO_RUN}_fit_summary.json'

fan_estimates.to_csv(fan_csv, index=False)
summary_json.write_text(json.dumps(summary, indent=2))

fan_estimates.head(), summary
